# IT3091 Machine Learning — Telco Customer Churn
## Notebook 2: Data Preprocessing



## Step 1 — Load and inspect raw data

Read the original CSV without overwriting it. Inspect its shape, data types, missing-value counts, and target distribution. The dataset contains **7,043 customer records and 21 raw columns**. The target distribution is **5,174 `No` and 1,869 `Yes`**, so accuracy alone will not adequately measure how well a model identifies churners. A simple null check misses whitespace-only entries in `TotalCharges`.

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/Original/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

print("Dataset shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nChurn distribution:")
print(df["Churn"].value_counts())

Dataset shape: (7043, 21)

Data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Missing values:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract    

## Step 2 — Resolve hidden missing values in `TotalCharges`

The original `TotalCharges` field is text because **11 rows contain blank spaces**. Convert it to numeric using `errors="coerce"`, examine the affected customers, and set the resulting missing amounts to zero. All 11 affected customers have **tenure = 0 months**, making zero accumulated charges a reasonable documented business assumption. This correction is deterministic; it does not learn dataset-wide statistics. Do not automatically apply zero imputation to unrelated future missing records without checking their meaning.

In [2]:
# Step 2: Handle missing values in TotalCharges

# Convert TotalCharges from text to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Check missing values after conversion
print("Missing TotalCharges before filling:")
print(df["TotalCharges"].isnull().sum())

# Check tenure of customers with missing TotalCharges
print("\nTenure of affected customers:")
print(df.loc[df["TotalCharges"].isnull(), "tenure"].value_counts())

# Fill missing TotalCharges with 0
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# Verify the result
print("\nMissing TotalCharges after filling:")
print(df["TotalCharges"].isnull().sum())

print("\nNew data type:")
print(df["TotalCharges"].dtype)

Missing TotalCharges before filling:
11

Tenure of affected customers:
tenure
0    11
Name: count, dtype: int64

Missing TotalCharges after filling:
0

New data type:
float64


## Step 3 — Define predictors and classification target

Exclude `customerID` because it is a unique identifier, and exclude `Churn` from the input features to prevent target leakage. Convert the binary target to `0` (stayed) and `1` (churned). This leaves **19 original predictors**.

In [3]:
# Step 3: Separate features and target

# Remove customerID and Churn from input features
X = df.drop(columns=["customerID", "Churn"])

# Convert target into binary values
# Yes = 1 (Churned), No = 0 (Stayed)
y = df["Churn"].map({"Yes": 1, "No": 0})

# Verify
print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nFeature columns:")
print(X.columns.tolist())

Feature shape: (7043, 19)
Target shape: (7043,)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Feature columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


## Step 4 — Make one shared stratified train/test split

Reserve **20% of customers for final testing**, with `random_state=42` for reproducibility and `stratify=y` to retain approximately the same churn proportions in both splits. The resulting datasets contain **5,634 training** and **1,409 test** records. Everyone in the group should reuse this split. **Do not use the test set to select features, tune models, thresholds, or compare candidates.**

In [4]:
# Step 4: Train/Test Split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

print("\nTraining target distribution (%):")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution (%):")
print(y_test.value_counts(normalize=True) * 100)

X_train shape: (5634, 19)
X_test shape: (1409, 19)

Training target distribution (%):
Churn
0    73.464679
1    26.535321
Name: proportion, dtype: float64

Testing target distribution (%):
Churn
0    73.456352
1    26.543648
Name: proportion, dtype: float64


## Step 5 — Encode categorical variables (learning demonstration)

The training set contains **15 text-based categorical columns**. `OneHotEncoder` expands these to **41 indicator columns** while preserving categories such as `No internet service` as distinct values. Fit the encoder on `X_train` only; reuse it for `X_test`. Unknown future categories are ignored rather than causing transformation errors. This manual encoding demonstrates the process; **Step 8 defines the authoritative reusable transformer**.

In [5]:
# Step 5: Categorical Encoding

from sklearn.preprocessing import OneHotEncoder

# Identify categorical columns
categorical_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical columns:")
print(categorical_cols)

# Create encoder
encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# Fit on training data only
X_train_encoded = encoder.fit_transform(
    X_train[categorical_cols]
)

# Transform test data using the same encoder
X_test_encoded = encoder.transform(
    X_test[categorical_cols]
)

print("\nEncoded training shape:", X_train_encoded.shape)
print("Encoded testing shape:", X_test_encoded.shape)

print("\nFirst 10 encoded feature names:")
print(encoder.get_feature_names_out(categorical_cols)[:10])

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']

Encoded training shape: (5634, 41)
Encoded testing shape: (1409, 41)

First 10 encoded feature names:
['gender_Female' 'gender_Male' 'Partner_No' 'Partner_Yes' 'Dependents_No'
 'Dependents_Yes' 'PhoneService_No' 'PhoneService_Yes' 'MultipleLines_No'
 'MultipleLines_No phone service']


## Step 6 — Scale continuous values and combine features (learning demonstration)

Use `StandardScaler` for `tenure`, `MonthlyCharges`, and `TotalCharges`. It subtracts the **training mean** and divides by the **training standard deviation**; negative standardized values are valid. Keep `SeniorCitizen` as its original 0/1 indicator. Combine these 3 scaled columns, 1 binary column, and 41 encoded columns into **45 numerical features**. Step 8 reproduces this behavior in a reusable transformer; **do not fit these hand-built arrays again during cross-validation**.

In [6]:
# Step 6: Scale numeric features and combine all features

from sklearn.preprocessing import StandardScaler

numeric_cols = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

binary_cols = ["SeniorCitizen"]

# Create scaler
scaler = StandardScaler()

# Fit only on training data
X_train_scaled = scaler.fit_transform(
    X_train[numeric_cols]
)

# Apply same scaling to test data
X_test_scaled = scaler.transform(
    X_test[numeric_cols]
)

# Create DataFrames with matching indexes
train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=numeric_cols,
    index=X_train.index
)

test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=numeric_cols,
    index=X_test.index
)

train_encoded_df = pd.DataFrame(
    X_train_encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_train.index
)

test_encoded_df = pd.DataFrame(
    X_test_encoded,
    columns=encoder.get_feature_names_out(categorical_cols),
    index=X_test.index
)

# Combine encoded, scaled and binary features
X_train_final = pd.concat(
    [
        train_scaled_df,
        X_train[binary_cols],
        train_encoded_df
    ],
    axis=1
)

X_test_final = pd.concat(
    [
        test_scaled_df,
        X_test[binary_cols],
        test_encoded_df
    ],
    axis=1
)

# Verify
print("Final training shape:", X_train_final.shape)
print("Final testing shape:", X_test_final.shape)

print("\nMissing values in training:",
      X_train_final.isnull().sum().sum())

print("Missing values in testing:",
      X_test_final.isnull().sum().sum())

print("\nFirst 5 rows:")
print(X_train_final.head())

Final training shape: (5634, 45)
Final testing shape: (1409, 45)

Missing values in training: 0
Missing values in testing: 0

First 5 rows:
        tenure  MonthlyCharges  TotalCharges  SeniorCitizen  gender_Female  \
3738  0.102371       -0.521976     -0.262257              0            0.0   
3151 -0.711743        0.337478     -0.503635              0            0.0   
4860 -0.793155       -0.809013     -0.749883              0            0.0   
3867 -0.263980        0.284384     -0.172722              0            1.0   
3810 -1.281624       -0.676279     -0.989374              0            0.0   

      gender_Male  Partner_No  Partner_Yes  Dependents_No  Dependents_Yes  \
3738          1.0         1.0          0.0            1.0             0.0   
3151          1.0         0.0          1.0            0.0             1.0   
4860          1.0         0.0          1.0            0.0             1.0   
3867          0.0         0.0          1.0            1.0             0.0   
3810  

## Step 7 — Assess class imbalance without modifying records

The training subset contains **4,139 non-churners (73.46%)** and **1,495 churners (26.54%)**. Calculate balanced class weights as a potential modeling option: approximately **0.6806 for class 0** and **1.8843 for class 1**. This is an assessment only: **no resampling or class weighting has been applied to a trained model**. Later, compare ordinary and class-weighted classifiers using recall, precision, F1, and ROC-AUC/PR-based measures as appropriate.

In [7]:
# Step 7: Analyze training class imbalance

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Training distribution
print("Training class counts:")
print(y_train.value_counts())

print("\nTraining class percentages:")
print((y_train.value_counts(normalize=True) * 100).round(2))

# Calculate balanced class weights
classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))

print("\nCalculated class weights:")
print(class_weights)

print("\nTest class distribution (unchanged):")
print(y_test.value_counts())

Training class counts:
Churn
0    4139
1    1495
Name: count, dtype: int64

Training class percentages:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64

Calculated class weights:
{np.int64(0): np.float64(0.6805991785455424), np.int64(1): np.float64(1.8842809364548494)}

Test class distribution (unchanged):
Churn
0    1035
1     374
Name: count, dtype: int64


## Step 8 — Define the reusable preprocessing transformer

`ColumnTransformer` standardizes three continuous variables, one-hot encodes 15 categorical variables, and passes through `SeniorCitizen`, yielding **45 features**. Fit on the training split only and transform the test split using the fitted object. Unlike the earlier manual exercise, **this transformer is the reusable artifact for the team**.

**Important cross-validation rule:** to avoid validation-fold leakage, each teammate should put a **fresh/unfitted** `ColumnTransformer` inside their model’s `Pipeline` before calling cross-validation or hyperparameter search on the **raw training split**. A transformer already fitted on all 5,634 training rows must not be reused inside training-fold validation.

In [8]:
# Step 8: Reusable Preprocessing Pipeline

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# Define feature groups
numeric_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

binary_features = ["SeniorCitizen"]

categorical_features = categorical_cols

# Build preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ), categorical_features),
        ("binary", "passthrough", binary_features)
    ],
    remainder="drop"
)

# Fit only on training data
X_train_processed = preprocessor.fit_transform(X_train)

# Transform test data without refitting
X_test_processed = preprocessor.transform(X_test)

# Verify
print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

print("\nMissing training values:",
      np.isnan(X_train_processed).sum())

print("Missing testing values:",
      np.isnan(X_test_processed).sum())

print("\nFeature count:",
      len(preprocessor.get_feature_names_out()))

Processed training shape: (5634, 45)
Processed testing shape: (1409, 45)

Missing training values: 0
Missing testing values: 0

Feature count: 45


## Step 9 — Validate and export shared artifacts

Validate dimensions, columns, missing values, target alignment, and disjoint train/test row indices. Save the transformed training and testing feature tables, their labels, the **raw/untransformed feature splits**, and the fitted `preprocessor.joblib`. Keep the same row order across X and y.

Use the **raw splits + an unfitted preprocessing/model pipeline** for cross-validation. Use the prepared test split only for the final, once-per-selected-model evaluation. `joblib` files should be loaded only from trusted sources.

In [9]:
# Step 9: Validate and save preprocessing outputs

from pathlib import Path
import joblib
import pandas as pd
import numpy as np

# Create output directories
output_dir = Path("../data/processed")
model_dir = Path("../models")

output_dir.mkdir(parents=True, exist_ok=True)
model_dir.mkdir(parents=True, exist_ok=True)

# Get the transformed feature names
feature_names = preprocessor.get_feature_names_out()

# Convert arrays to DataFrames
train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

# Validate processed datasets
assert train_processed_df.shape[1] == test_processed_df.shape[1]
assert train_processed_df.columns.equals(test_processed_df.columns)
assert not train_processed_df.isna().any().any()
assert not test_processed_df.isna().any().any()
assert len(train_processed_df) == len(y_train)
assert len(test_processed_df) == len(y_test)
assert set(X_train.index).isdisjoint(set(X_test.index))

# Save transformed features
train_processed_df.to_csv(
    output_dir / "X_train_processed.csv",
    index=False
)

test_processed_df.to_csv(
    output_dir / "X_test_processed.csv",
    index=False
)

# Save target variables
y_train.to_csv(output_dir / "y_train.csv", index=False)
y_test.to_csv(output_dir / "y_test.csv", index=False)

# Save original untransformed splits
X_train.to_csv(output_dir / "X_train_raw.csv", index=False)
X_test.to_csv(output_dir / "X_test_raw.csv", index=False)

# Save fitted preprocessor
joblib.dump(
    preprocessor,
    model_dir / "preprocessor.joblib"
)

print("All preprocessing outputs saved successfully.")

print("\nTraining features:", train_processed_df.shape)
print("Testing features:", test_processed_df.shape)

print("\nSaved feature count:", len(feature_names))

All preprocessing outputs saved successfully.

Training features: (5634, 45)
Testing features: (1409, 45)

Saved feature count: 45


## Step 10 — Final verification

Confirm **7,043 × 21** raw data, **5,634 × 45** training data, **1,409 × 45** test data, zero missing transformed values, **zero overlap** between splits, and consistent feature counts. These checks establish internal consistency, but actual modeling performance still needs to be measured independently.

In [10]:
# Step 10: Final Preprocessing Validation

import pandas as pd
import numpy as np

print("=== FINAL PREPROCESSING SUMMARY ===")

print("\nOriginal dataset:", df.shape)
print("Training dataset:", X_train_processed.shape)
print("Testing dataset:", X_test_processed.shape)

print("\nTarget distribution:")
print(y.value_counts())

print("\nMissing values:")
print("Training:", np.isnan(X_train_processed).sum())
print("Testing:", np.isnan(X_test_processed).sum())

print("\nDuplicate customer overlap:")
print(len(set(X_train.index) & set(X_test.index)))

print("\nFeature names consistent:")
print(
    X_train_processed.shape[1] ==
    X_test_processed.shape[1] ==
    len(preprocessor.get_feature_names_out())
)

print("\nTraining target counts:")
print(y_train.value_counts())

print("\nTesting target counts:")
print(y_test.value_counts())

=== FINAL PREPROCESSING SUMMARY ===

Original dataset: (7043, 21)
Training dataset: (5634, 45)
Testing dataset: (1409, 45)

Target distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64

Missing values:
Training: 0
Testing: 0

Duplicate customer overlap:
0

Feature names consistent:
True

Training target counts:
Churn
0    4139
1    1495
Name: count, dtype: int64

Testing target counts:
Churn
0    1035
1     374
Name: count, dtype: int64


## Preprocessing decision log

| Decision | Justification | Verified outcome / status |
|---|---|---|
| Convert `TotalCharges` to numeric | Monetary amounts should be numeric, not text | Converted to `float64` |
| Fill 11 blank `TotalCharges` with 0 | All 11 have tenure 0; zero accumulated billing is reasonable | No missing values remained in this field |
| Drop `customerID` from predictors | Unique ID is not a service or behavior attribute | 19 remaining input variables |
| Keep `Churn` out of features, encode as 0/1 | Avoid target leakage; express binary classification label | 5,174 class 0 and 1,869 class 1 overall |
| 80/20 stratified split (`random_state=42`) | Reproducibility and comparable class ratios | 5,634 train, 1,409 test |
| One-hot encode service and contract categories | Avoid artificial numeric ordering | 15 categorical inputs → 41 indicator columns |
| Keep `No internet service` and `No phone service` separate | These encode unavailable service, not simply an unpurchased add-on | Original distinctions retained |
| Standardize tenure and charge amounts | Consistent feature scales for scale-sensitive models | Three scaled inputs using training-set statistics |
| Keep `SeniorCitizen` as 0/1 | Already a binary numeric feature | Passed through unchanged |
| Investigate balanced class weights | Only 26.54% of training records are churners | 0: ~0.6806; 1: ~1.8843; **not yet applied** |
| Use `ColumnTransformer` | Reproducible numeric/categorical processing | 45 features; no missing transformed values |
| Export raw and transformed splits | Support shared model training and leakage-safe CV | Saved CSVs and fitted preprocessor |

### Leakage prevention and modeling handoff

- The **test set is held out** and must not guide model selection, threshold selection, or hyperparameter tuning.
- Scale/encode only after splitting, and **fit** each learned transformation on the corresponding training data.
- During cross-validation, start from **`X_train_raw.csv`** and use a **fresh** preprocessor inside an estimator `Pipeline` so every fold fits its own transformations. Do **not** cross-validate on `X_train_processed.csv`, because it was transformed using statistics learned from the whole training split.
- `X_train_processed.csv` is useful for quick fixed-split experimentation, but pipeline-based modeling is preferred for consistent validation and later inference.
- Class weights are an option to compare during training; **SMOTE/oversampling was not performed**.
- Results show associations with churn, **not proof of causality**.

### Handoff summary

The initial feature set comprises **19 source predictors → 45 transformed numerical columns**. The shared split uses `random_state=42`, with **4,139 / 1,495** class counts in training and **1,035 / 374** in testing (No/Yes). Training/test preprocessing is consistent, and all exported arrays were validated for missing values and overlap. Next: train and cross-validate candidate classifiers, compare recall, precision, F1, ROC-AUC and PR performance, then reserve the held-out test set for the chosen model’s final assessment.
